<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/11.%20Novelty%20and%20Outlier%20Detection/ch11_outlier_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 11: Novelty and Outlier Detection

## Anomaly Detection

Anomaly detection mengidentifikasi titik data yang **berbeda secara signifikan** dari pola mayoritas data.

### Dua Skenario Utama

1. **Outlier Detection**: Training data mengandung outlier; model harus tahan terhadap kontaminasi
2. **Novelty Detection**: Training data bersih (hanya normal data); deteksi titik baru yang tidak fit dengan distribusi normal

### Aplikasi
- Deteksi fraud transaksi keuangan
- Deteksi intrusi jaringan (cybersecurity)
- Quality control manufaktur
- Predictive maintenance (anomali sensor)
- Medical diagnosis (deteksi kondisi langka)

## Metode Deteksi Anomali

| Method | Pendekatan | Training |
|---|---|---|
| **Isolation Forest** | Tree-based; isolasi outlier dengan sedikit split | Unsupervised |
| **Local Outlier Factor** | Density-based; bandingkan densitas lokal | Unsupervised |
| **One-Class SVM** | Boundary-based; pisahkan data normal dari asal | Semi-supervised |
| **Elliptic Envelope** | Statistical; fit ellipse di sekitar data normal | Unsupervised |
| **DBSCAN** | Density-based clustering; noise = outlier | Unsupervised |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_breast_cancer
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Generate normal + outlier data
np.random.seed(42)
n_normal = 300
n_outliers = 30

X_normal = 0.3 * np.random.randn(n_normal, 2)
X_outliers = np.random.uniform(low=-4, high=4, size=(n_outliers, 2))
X_all = np.vstack([X_normal, X_outliers])
y_all = np.array([1] * n_normal + [-1] * n_outliers)  # 1=normal, -1=outlier

print(f'Normal samples: {n_normal}')
print(f'Outlier samples: {n_outliers}')
print(f'Contamination ratio: {n_outliers/(n_normal+n_outliers):.1%}')


## 11.1 Isolation Forest

Isolation Forest bekerja dengan prinsip bahwa **outlier lebih mudah diisolasi** daripada data normal:

- Anomali memiliki lebih sedikit "pertanyaan" yang diperlukan untuk mengisolasi mereka (path length pendek)
- Data normal membutuhkan banyak split untuk diisolasi (path length panjang)

**Anomaly Score**: rata-rata normalized path length dari semua trees:

$$s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}$$

di mana $c(n) = 2H(n-1) - \frac{2(n-1)}{n}$ adalah rata-rata path length untuk dataset berukuran $n$, dan $H$ adalah harmonic number.

Score mendekati 1 → anomali; mendekati 0 → normal.


In [ ]:
# Isolation Forest
contamination = n_outliers / (n_normal + n_outliers)
iforest = IsolationForest(contamination=contamination, random_state=42, n_estimators=100)
preds_if = iforest.fit_predict(X_all)
scores_if = iforest.score_samples(X_all)

# LOF
lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination)
preds_lof = lof.fit_predict(X_all)
scores_lof = -lof.negative_outlier_factor_

# One-Class SVM (fit on normal data only)
X_train_normal = X_normal
ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=contamination)
ocsvm.fit(X_train_normal)
preds_ocsvm = ocsvm.predict(X_all)

# Elliptic Envelope
ellenv = EllipticEnvelope(contamination=contamination, random_state=42)
preds_ee = ellenv.fit_predict(X_all)

# Results
print('=== Anomaly Detection Results ===')
print(f'{"Method":<25} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 57)
for name, preds in [
    ('Isolation Forest', preds_if),
    ('LOF', preds_lof),
    ('One-Class SVM', preds_ocsvm),
    ('Elliptic Envelope', preds_ee)
]:
    from sklearn.metrics import precision_recall_fscore_support
    p, r, f1, _ = precision_recall_fscore_support(y_all, preds, labels=[-1], zero_division=0)
    print(f'{name:<25} {p[0]:>10.4f} {r[0]:>10.4f} {f1[0]:>10.4f}')


In [ ]:
# Visualization: decision boundaries
def plot_anomaly_boundary(ax, X, y_true, detector, title, resolution=200):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    grid = np.c_[xx.ravel(), yy.ravel()]
    try:
        Z = detector.score_samples(grid)
    except:
        Z = detector.decision_function(grid)
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=20, cmap='RdYlGn', alpha=0.5)
    ax.contour(xx, yy, Z, levels=[Z.mean()], colors='black', linewidths=2)
    normal = X[y_true == 1]
    outlier = X[y_true == -1]
    ax.scatter(normal[:, 0], normal[:, 1], c='blue', s=15, alpha=0.5, label='Normal')
    ax.scatter(outlier[:, 0], outlier[:, 1], c='red', s=60, marker='x', label='Outlier', zorder=5)
    ax.set_title(title)
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_anomaly_boundary(axes[0], X_all, y_all, iforest, 'Isolation Forest')
plot_anomaly_boundary(axes[1], X_all, y_all, ocsvm, 'One-Class SVM')
plot_anomaly_boundary(axes[2], X_all, y_all, ellenv, 'Elliptic Envelope')
plt.tight_layout()
plt.show()


## 11.2 Local Outlier Factor (LOF)

LOF mengukur **deviasi densitas lokal** suatu titik relatif terhadap tetangganya:

$$LOF_k(p) = \frac{\sum_{o \in N_k(p)} \frac{lrd_k(o)}{lrd_k(p)}}{|N_k(p)|}$$

di mana **Local Reachability Density**:

$$lrd_k(p) = \frac{|N_k(p)|}{\sum_{o \in N_k(p)} \text{reach-dist}_k(p, o)}$$

- $LOF \approx 1$: densitas lokal titik mirip dengan tetangganya → **normal**
- $LOF >> 1$: densitas lokal titik jauh lebih rendah dari tetangganya → **outlier**


In [ ]:
# LOF Visualization: score distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# LOF scores distribution
axes[0].hist(scores_lof[y_all == 1], bins=30, alpha=0.6, color='blue', label='Normal', density=True)
axes[0].hist(scores_lof[y_all == -1], bins=15, alpha=0.6, color='red', label='Outlier', density=True)
axes[0].set_xlabel('LOF Score (higher = more anomalous)')
axes[0].set_ylabel('Density')
axes[0].set_title('LOF Score Distribution')
axes[0].legend()

# Isolation Forest scores distribution
axes[1].hist(scores_if[y_all == 1], bins=30, alpha=0.6, color='blue', label='Normal', density=True)
axes[1].hist(scores_if[y_all == -1], bins=15, alpha=0.6, color='red', label='Outlier', density=True)
axes[1].set_xlabel('Anomaly Score (lower = more anomalous)')
axes[1].set_ylabel('Density')
axes[1].set_title('Isolation Forest Score Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

# AUC comparison
auc_if = roc_auc_score(np.where(y_all==-1, 1, 0), -scores_if)
auc_lof = roc_auc_score(np.where(y_all==-1, 1, 0), scores_lof)
print(f'Isolation Forest AUC: {auc_if:.4f}')
print(f'LOF AUC: {auc_lof:.4f}')


In [ ]:
# Effect of contamination parameter on Isolation Forest
contam_values = [0.01, 0.05, 0.10, 0.15, 0.20]
fig, axes = plt.subplots(1, 5, figsize=(20, 3))

for ax, contam in zip(axes, contam_values):
    ifo = IsolationForest(contamination=contam, random_state=42, n_estimators=100)
    preds = ifo.fit_predict(X_all)
    n_detected = np.sum(preds == -1)
    tp = np.sum((preds == -1) & (y_all == -1))
    fp = np.sum((preds == -1) & (y_all == 1))

    ax.scatter(X_all[preds==1, 0], X_all[preds==1, 1], c='blue', s=10, alpha=0.4, label='Normal')
    ax.scatter(X_all[preds==-1, 0], X_all[preds==-1, 1], c='red', s=30, marker='x', label='Anomaly')
    ax.set_title(f'contam={contam}\nTP={tp}, FP={fp}')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


## Ringkasan: Key Takeaways Chapter 11

| Method | Kelebihan | Kekurangan | Hyperparameter Utama |
|---|---|---|---|
| **Isolation Forest** | Cepat, scalable, robust | Tidak bekerja baik untuk cluster tinggi | `contamination`, `n_estimators` |
| **LOF** | Deteksi outlier lokal, kontekstual | Lambat untuk data besar ($O(n^2)$) | `n_neighbors`, `contamination` |
| **One-Class SVM** | Flexible dengan kernel | Lambat, sensitif terhadap skala | `nu`, `kernel`, `gamma` |
| **Elliptic Envelope** | Robust untuk distribusi Gaussian | Asumsi Gaussian, tidak efektif untuk non-Gaussian | `contamination` |

### Panduan Praktis
1. **Mulai dengan Isolation Forest** — cepat, efektif, sedikit asumsi distribusi
2. **LOF** untuk anomali yang bersifat **lokal/kontekstual** (normal di satu region, outlier di region lain)
3. **One-Class SVM** ketika ada data normal yang bersih untuk training
4. **Selalu evaluasi** dengan domain knowledge — "outlier" bergantung pada konteks bisnis
5. **Contamination** adalah hyperparameter terpenting — estimasi dari domain knowledge
